# N-Back Task Tutorial: Running and Editing a Pavlovia Experiment

This notebook walks you through a real N-back cognitive task that was created in PsychoPy and hosted on Pavlovia. The code has been broken into **sections** so you can:

1. **Understand** what each part of the experiment does.
2. **Run** each section independently to see its effect.
3. **Edit** specific parameters — like the colour of the square or the value of N — to see how the task changes.

---

## What is the N-Back Task?

The N-back task is a cognitive test that measures **working memory**. A sequence of stimuli (here, coloured squares in different positions) is shown one at a time. The participant presses **Space** whenever the current stimulus matches the one that appeared **N trials ago**.

- **2-back**: match with the item 2 positions back.
- **3-back**: match with the item 3 positions back (harder!).

---

## Requirements

Make sure you are running this notebook with the **`psychopy_env`** kernel. The following packages are needed:

```
psychopy
matplotlib
pandas
```

> **Note:** Cells that open a PsychoPy window require a display (run on your local machine, not a remote server).

---
## Section 1 — Imports

Run this cell first. It loads all the Python packages the experiment needs.

In [ ]:
# Section 1: Imports
# ---------------------------------------------------------
# These are the same libraries used in the Pavlovia version
# of the experiment.
# ---------------------------------------------------------

from psychopy import visual, core, event, data, logging
import random
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches

print("All libraries loaded successfully!")

---
## Section 2 — Experiment Parameters

This is where you set the **editable parameters** for the task.

| Parameter | Description | Default |
|-----------|-------------|---------|
| `N_BACK` | How many trials back to compare | `2` |
| `SQUARE_COLOR` | Fill colour of the square | `'blue'` |
| `N_TRIALS` | Number of stimuli in the sequence | `20` |
| `MATCH_PROBABILITY` | Probability that a trial is an N-back match | `0.30` |
| `STIMULUS_DURATION` | How long the square is shown (seconds) | `0.5` |
| `ISI_DURATION` | Inter-stimulus interval (seconds) | `1.0` |

> ✏️ **Try it yourself!** Change `SQUARE_COLOR` or `N_BACK` in the cell below and re-run it.

In [ ]:
# Section 2: Experiment Parameters
# ---------------------------------------------------------
# Edit the values below to customise the task.
# ---------------------------------------------------------

# ✏️ EXERCISE A — change the color here (e.g. 'red', 'green', 'yellow', 'orange')
SQUARE_COLOR = 'blue'

# ✏️ EXERCISE B — change 2 to 3 for a 3-back task
N_BACK = 2

# Other task parameters
N_TRIALS           = 20     # total number of stimuli
MATCH_PROBABILITY  = 0.30   # ~30 % of trials are N-back matches
STIMULUS_DURATION  = 0.5    # seconds the square is shown
ISI_DURATION       = 1.0    # inter-stimulus interval in seconds

print(f"Parameters set:")
print(f"  N-back level    : {N_BACK}")
print(f"  Square color   : {SQUARE_COLOR}")
print(f"  Number of trials: {N_TRIALS}")

---
## Section 3 — Preview the Stimulus

Before opening the PsychoPy window, use **Matplotlib** to preview what the stimulus will look like with your chosen colour. This is helpful for quickly checking colour changes without having to run the full experiment.

In [ ]:
# Section 3: Stimulus Preview (Matplotlib)
# ---------------------------------------------------------
# Shows a quick visual preview of the square stimulus
# using the parameters set in Section 2.
# ---------------------------------------------------------

fig, ax = plt.subplots(figsize=(4, 4))
ax.set_facecolor('#808080')          # grey background (same as PsychoPy default)
fig.patch.set_facecolor('#808080')

# Draw the fixation cross
ax.text(0, 0, '+', color='white', fontsize=20, ha='center', va='center')

# Draw the N-back square at the top-left position as an example
square = patches.FancyBboxPatch(
    (-0.15, 0.10), 0.30, 0.30,
    boxstyle="square,pad=0",
    linewidth=2, edgecolor='white', facecolor=SQUARE_COLOR
)
ax.add_patch(square)

ax.set_xlim(-0.6, 0.6)
ax.set_ylim(-0.6, 0.6)
ax.set_aspect('equal')
ax.axis('off')
ax.set_title(
    f"{N_BACK}-Back Task  |  Square color: {SQUARE_COLOR}",
    color='white', pad=10
)
plt.tight_layout()
plt.show()

---
## Section 4 — Generate the Trial Sequence

This section builds the list of stimuli. Each stimulus is a **screen position** (one of four locations). A proportion of trials (`MATCH_PROBABILITY`) are intentionally set to match the position shown N trials earlier.

In [ ]:
# Section 4: Generate the Trial Sequence
# ---------------------------------------------------------
# Creates a randomised sequence of stimulus positions and
# marks which trials are N-back matches.
# ---------------------------------------------------------

random.seed(42)   # fix the seed so results are reproducible

# Four possible positions on screen (x, y) in PsychoPy normalised units
POSITIONS = [
    (-0.3,  0.3),   # top-left
    ( 0.3,  0.3),   # top-right
    (-0.3, -0.3),   # bottom-left
    ( 0.3, -0.3),   # bottom-right
]

# Start with a fully random sequence
sequence = [random.choice(POSITIONS) for _ in range(N_TRIALS)]

# Overwrite some trials to create N-back matches
for i in range(N_BACK, N_TRIALS):
    if random.random() < MATCH_PROBABILITY:
        sequence[i] = sequence[i - N_BACK]

# Build a summary table
rows = []
for i, pos in enumerate(sequence):
    is_match = (i >= N_BACK) and (pos == sequence[i - N_BACK])
    rows.append({'Trial': i + 1, 'Position': pos, 'N-back Match': is_match})

df_sequence = pd.DataFrame(rows)
print(f"Sequence generated: {N_TRIALS} trials, "
      f"{df_sequence['N-back Match'].sum()} N-back matches "
      f"({df_sequence['N-back Match'].mean()*100:.0f}%)")
df_sequence.head(10)

---
## Section 5 — PsychoPy Window & Stimuli Setup

This section creates the PsychoPy **Window** and the **Rect** stimulus (the coloured square). It uses the `SQUARE_COLOR` you set in Section 2.

> ⚠️ Running this cell will open a grey PsychoPy window on your screen.  
> Make sure you are using the **`psychopy_env`** kernel.

In [ ]:
# Section 5: PsychoPy Window & Stimuli Setup
# ---------------------------------------------------------
# Opens the experiment window and creates the visual
# objects (square, fixation cross, feedback text).
# ---------------------------------------------------------

win = visual.Window(
    size=[800, 600],
    fullscr=False,
    color='gray',
    units='norm',
    title=f"{N_BACK}-Back Task"
)

# The colored square — SQUARE_COLOR is set in Section 2
square_stim = visual.Rect(
    win,
    width=0.30,
    height=0.30,
    fillColor=SQUARE_COLOR,   # ← this is the editable color
    lineColor=SQUARE_COLOR
)

# Fixation cross shown between trials
fixation_stim = visual.TextStim(
    win, text='+', color='white', height=0.10
)

# Text used for instructions and feedback
instruction_stim = visual.TextStim(
    win, text='', color='white', height=0.07, wrapWidth=1.8
)

print("PsychoPy window opened successfully.")
print(f"Square color is set to: {SQUARE_COLOR}")

---
## Section 6 — Instructions Screen

Displays the task instructions inside the PsychoPy window and waits for the participant to press **Space** to start.

In [ ]:
# Section 6: Instructions Screen
# ---------------------------------------------------------
# Shows instructions and waits for the participant to
# press Space before the task begins.
# ---------------------------------------------------------

instruction_stim.text = (
    f"{N_BACK}-Back Task\n\n"
    f"Press SPACE whenever the square appears\n"
    f"in the SAME position as {N_BACK} trial(s) ago.\n\n"
    f"Press SPACE to start."
)
instruction_stim.draw()
win.flip()

# Wait until the participant presses Space
event.waitKeys(keyList=['space'])
print("Instructions acknowledged. Task starting...")

---
## Section 7 — Run the Task

This is the **main trial loop**. For each trial it:

1. Shows a fixation cross for `ISI_DURATION` seconds.
2. Moves the square to the trial's position and shows it for `STIMULUS_DURATION` seconds.
3. Collects a Space-bar response during the stimulus display.
4. Records whether the response was correct.

In [ ]:
# Section 7: Main Trial Loop
# ---------------------------------------------------------
# Runs through the sequence generated in Section 4 and
# records responses.
# ---------------------------------------------------------

results = []
clock = core.Clock()

for trial_num, pos in enumerate(sequence):
    # --- Fixation ---
    fixation_stim.draw()
    win.flip()
    core.wait(ISI_DURATION)

    # --- Stimulus ---
    square_stim.pos = pos
    square_stim.draw()
    win.flip()

    # Collect response during stimulus display
    event.clearEvents()
    clock.reset()
    keys = event.waitKeys(
        maxWait=STIMULUS_DURATION,
        keyList=['space', 'escape'],
        timeStamped=clock
    )

    # Allow participant to quit at any time
    if keys and any(k == 'escape' for k, _ in keys):
        print("Task aborted by participant.")
        break

    # Determine correctness
    is_match   = (trial_num >= N_BACK) and (pos == sequence[trial_num - N_BACK])
    responded  = keys is not None and any(k == 'space' for k, _ in keys)
    rt         = keys[0][1] if responded else None
    correct    = (is_match and responded) or (not is_match and not responded)

    results.append({
        'trial'     : trial_num + 1,
        'position'  : pos,
        'is_match'  : is_match,
        'responded' : responded,
        'rt'        : rt,
        'correct'   : correct,
    })

    # --- Blank inter-trial interval ---
    win.flip()
    core.wait(0.1)

print(f"Task complete — {len(results)} trials recorded.")

---
## Section 8 — Close the Window

In [ ]:
# Section 8: Close the PsychoPy Window
# ---------------------------------------------------------
# Always close the window cleanly after the task.
# ---------------------------------------------------------

win.close()
print("Window closed.")

---
## Section 9 — Analyse the Results

Summarise accuracy and reaction times from the trial data collected in Section 7.

In [ ]:
# Section 9: Results Analysis
# ---------------------------------------------------------
# Summarises accuracy and plots reaction times.
# ---------------------------------------------------------

df_results = pd.DataFrame(results)

if df_results.empty:
    print("No results to analyse — run Sections 5-7 first.")
else:
    n_correct  = df_results['correct'].sum()
    n_total    = len(df_results)
    accuracy   = n_correct / n_total * 100

    hit_rate   = df_results.loc[df_results['is_match'],  'responded'].mean() * 100
    fa_rate    = df_results.loc[~df_results['is_match'], 'responded'].mean() * 100

    print(f"Overall accuracy : {n_correct}/{n_total} = {accuracy:.1f}%")
    print(f"Hit rate         : {hit_rate:.1f}%  (correct presses on match trials)")
    print(f"False alarm rate : {fa_rate:.1f}%  (incorrect presses on non-match trials)")

    # Plot RT distribution for correct responses
    correct_rts = df_results.loc[df_results['correct'] & df_results['rt'].notna(), 'rt']
    if not correct_rts.empty:
        fig, ax = plt.subplots(figsize=(6, 3))
        ax.hist(correct_rts, bins=10, color=SQUARE_COLOR, edgecolor='black')
        ax.set_xlabel("Reaction Time (s)")
        ax.set_ylabel("Count")
        ax.set_title(f"Correct Response RT Distribution ({N_BACK}-Back, {SQUARE_COLOR} square)")
        plt.tight_layout()
        plt.show()

    display(df_results)

---
## ✏️ Exercise 1 — Change the Square Colour

**Goal:** Change the colour of the square from **blue** to any colour you like.

1. Go back to **Section 2** and change `SQUARE_COLOR = 'blue'` to another colour (e.g. `'red'`, `'green'`, `'yellow'`, `'orange'`, `'purple'`).
2. Re-run **Section 2** (the parameter cell).
3. Re-run **Section 3** (the preview cell) to immediately see the new colour in the Matplotlib preview — *no PsychoPy window needed!*
4. (Optional) Re-run Sections 5–8 to see the colour change in the actual experiment.

The cell below is a standalone version for quick experimentation:

In [ ]:
# ✏️ Exercise 1: Change the square color
# ---------------------------------------------------------
# Edit SQUARE_COLOR below, then run this cell.
# ---------------------------------------------------------

SQUARE_COLOR = 'red'   # ← change this value!

# Quick preview
fig, ax = plt.subplots(figsize=(3, 3))
ax.set_facecolor('#808080')
fig.patch.set_facecolor('#808080')
square = patches.FancyBboxPatch(
    (-0.15, -0.15), 0.30, 0.30,
    boxstyle="square,pad=0",
    linewidth=2, edgecolor='white', facecolor=SQUARE_COLOR
)
ax.add_patch(square)
ax.set_xlim(-0.6, 0.6)
ax.set_ylim(-0.6, 0.6)
ax.set_aspect('equal')
ax.axis('off')
ax.set_title(f'Square color: {SQUARE_COLOR}', color='white')
plt.tight_layout()
plt.show()

print(f"Square color changed to: {SQUARE_COLOR}")
print("Re-run Sections 5-8 to use this color in the full experiment.")

---
## ✏️ Exercise 2 — Change from 2-Back to 3-Back

**Goal:** Make the task harder by increasing N from **2** to **3**.

In a 3-back task the participant must remember what appeared **3 trials ago** instead of 2. This places a higher demand on working memory.

1. Run the cell below to generate a 3-back sequence and see the difference.
2. To run the full 3-back task, go back to **Section 2**, set `N_BACK = 3`, and re-run Sections 2–8.

In [ ]:
# ✏️ Exercise 2: Change from 2-back to 3-back
# ---------------------------------------------------------
# Edit N_BACK_EX below (try 2, 3, or even 4!), then run
# this cell to see how the sequence changes.
# ---------------------------------------------------------

N_BACK_EX = 3   # ← change this value!

random.seed(42)
seq_ex = [random.choice(POSITIONS) for _ in range(N_TRIALS)]
for i in range(N_BACK_EX, N_TRIALS):
    if random.random() < MATCH_PROBABILITY:
        seq_ex[i] = seq_ex[i - N_BACK_EX]

rows_ex = []
for i, pos in enumerate(seq_ex):
    is_match = (i >= N_BACK_EX) and (pos == seq_ex[i - N_BACK_EX])
    rows_ex.append({'Trial': i + 1, 'Position': pos, 'N-back Match': is_match})

df_ex = pd.DataFrame(rows_ex)
n_matches = df_ex['N-back Match'].sum()
print(f"{N_BACK_EX}-Back sequence: {N_TRIALS} trials, {n_matches} matches ({n_matches/N_TRIALS*100:.0f}%)")
print()
print("First 10 trials:")
display(df_ex.head(10))

# Visualise which trials are matches
fig, ax = plt.subplots(figsize=(10, 2))
for i, row in df_ex.iterrows():
    color = SQUARE_COLOR if row['N-back Match'] else 'lightgrey'
    ax.add_patch(patches.Rectangle((i, 0), 0.8, 0.8, color=color, ec='black'))
    ax.text(i + 0.4, 0.4, str(i + 1), ha='center', va='center', fontsize=7)
ax.set_xlim(0, N_TRIALS)
ax.set_ylim(0, 1)
ax.axis('off')
ax.set_title(
    f"{N_BACK_EX}-Back Match Sequence  "
    f"(colored = match, grey = no match)"
)
plt.tight_layout()
plt.show()

print(f"\nTo run this as the full experiment, set N_BACK = {N_BACK_EX} in Section 2 and re-run Sections 2–8.")

---
## Summary

You have now seen how a Pavlovia N-back experiment is structured and how to customise it:

| What you changed | Where to edit | Effect |
|---|---|---|
| Square colour | `SQUARE_COLOR` in Section 2 | Changes the fill colour of the stimulus square |
| N value | `N_BACK` in Section 2 | Changes difficulty (2-back → 3-back etc.) |
| Number of trials | `N_TRIALS` in Section 2 | Makes the task longer or shorter |
| Match probability | `MATCH_PROBABILITY` in Section 2 | Controls how many trials are N-back matches |

All of these changes require only **one line of code** — the parameter definition in Section 2. This is exactly how you would edit the experiment before uploading a new version to Pavlovia.

---
*Next step: see the **Making** section of the Introduction Tutorial to learn how to build a new experiment from scratch.*